# Bank Distress Early-Warning Model — Feature Selection Workspace
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

Loads all five datasets into working DataFrames. Feature candidates come from the
literature synthesis (`../literature/features_by_dataset.md`); known data issues are
tracked in `data_concerns.md`.

| DataFrame | Contents | Grain |
|---|---|---|
| `financials` | quarterly call-report data, working column set | one row = one bank-quarter |
| `institutions` | bank directory: identity, charter, location, status | one row = one bank |
| `failures` | failed-bank list (ground truth for the label) | one row = one failure |
| `history` | structure changes: mergers, acquisitions, closures | one row = one event |
| `fred` | macro context: rates, unemployment, GDP, prices | one row = one quarter |

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("..") / "data" / "raw"

## Financials

Working set from the literature review: identifiers, capital (PCA label source), asset
quality, loan portfolio mix, liquidity/funding, profitability, plus the securities
amortized-cost-vs-fair-value fields that flagged SVB in the EDA case study.

In [10]:
FINANCIAL_COLS = [
    # identifiers
    "CERT", "REPDTE", "NAME", "ASSET", "DEP",
    # capital (label source: PCA ratios)
    "RBC1AAJ", "RBC1RWAJ", "RBCRWAJ", "RBCT1CER", "RBCT1J", "EQV", "EQ",
    # asset quality
    "NPERFV", "NCLNLSR", "NTLNLSR", "LNATRESR", "P3ASSETR", "P9ASSETR",
    "ORER", "ELNATRR",
    # loan portfolio mix (Cole & White 2012)
    "LNRECONSR", "LNRENRESR", "LNREMULTR", "LNRERESR", "IDNCCIR", "LNCONR",
    # liquidity / funding (incl. SVB-style signals)
    "DEPUNA", "ESTINS", "BRO", "BROR", "LNLSDEPR", "CHBALR",
    "SCHA", "SCHF", "SCAA", "SCAF",
    # profitability
    "ROA", "ROE", "NIMY", "EEFFR", "NETINC",
]

financial_files = sorted(DATA.glob("financials_*.parquet"))
financials = pd.concat(
    (pd.read_parquet(f, columns=FINANCIAL_COLS) for f in financial_files),
    ignore_index=True,
)
financials["REPDTE"] = pd.to_datetime(financials["REPDTE"])
financials = financials.sort_values(["CERT", "REPDTE"]).reset_index(drop=True)

print(f"financials: {financials.shape[0]:,} rows x {financials.shape[1]} cols "
      f"({len(financial_files)} quarters, {financials['REPDTE'].min():%Y-%m} to {financials['REPDTE'].max():%Y-%m})")

financials: 1,678,302 rows x 41 cols (169 quarters, 1984-03 to 2026-03)


## Institutions, failures, history, FRED

In [11]:
institutions = pd.read_parquet(DATA / "institutions.parquet")
failures = pd.read_parquet(DATA / "failures.parquet")
failures["FAILDATE"] = pd.to_datetime(failures["FAILDATE"])
history = pd.read_parquet(DATA / "history.parquet")
fred = pd.read_parquet(DATA / "fred.parquet")
fred["REPDTE"] = pd.to_datetime(fred["REPDTE"])

for name, df in [("institutions", institutions), ("failures", failures),
                 ("history", history), ("fred", fred)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

institutions: 27,836 rows x 149 cols
failures: 4,115 rows x 22 cols
history: 583,066 rows x 176 cols
fred: 175 rows x 13 cols


## Date cutoff: 1990Q1

The three PCA ratios are 0% populated through 1989 and 99.7% populated from 1990 on,
so nothing before 1990 can be labeled. We keep 1990-92 (834 failures, ~40% of all
usable positives) even though PCA legally took effect Dec 1992: the ratios are computed
identically, and we're predicting the financial condition, not the legal act — an
"as-if" classification, same approach as Cole & White's technical-failure construct.
Rationale and sources: `../literature/pca_label_definition.md`.

In [12]:
CUTOFF = "1990-01-01"

before = len(financials)
financials = financials[financials["REPDTE"] >= CUTOFF].reset_index(drop=True)
fred = fred[fred["REPDTE"] >= CUTOFF].reset_index(drop=True)

print(f"financials: {before:,} -> {len(financials):,} rows "
f"({financials['REPDTE'].min():%Y-%m} to {financials['REPDTE'].max():%Y-%m})")
print(f"fred: {len(fred)} quarters")

financials: 1,678,302 -> 1,258,888 rows (1990-03 to 2026-03)
fred: 147 quarters


## Join into one modeling table

`panel` = `financials` + `fred` (on quarter) + 4 static columns from `institutions` (on bank).

Kept separate on purpose:
- `failures` — contains the future (failure dates); joining it would leak the answer. Used only to validate the label after it's built.
- `history` — event-grain (mergers, renames); only needed later to classify why banks exit the panel.

In [13]:
# institutions: keep only the static features the literature supports
inst_cols = institutions[["CERT", "ESTYMD", "STALP", "BKCLASS", "REGAGNT"]].copy()
inst_cols["ESTYMD"] = pd.to_datetime(inst_cols["ESTYMD"], errors="coerce")

panel = (
    financials
    .merge(inst_cols, on="CERT", how="left")
    .merge(fred, on="REPDTE", how="left")
)

# bank age in years, as of each quarter
panel["AGE_YEARS"] = (panel["REPDTE"] - panel["ESTYMD"]).dt.days / 365.25

print(f"panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"institutions matched: {panel['STALP'].notna().mean():.1%}")
print(f"fred matched: {panel['UNRATE'].notna().mean():.1%}")

panel: 1,258,888 rows x 58 cols
institutions matched: 99.2%
fred matched: 100.0%


## `panel` is the working table — brainstorming and label build below

## Clean the CBLR zero-fill

For CBLR opt-in banks (2020+), the FDIC fills `RBCRWAJ` with **0.0** instead of null
while leaving `RBC1RWAJ` null. Those zeros are placeholders, not real ratios — left
in place they poison medians, distributions, and AUC scores. Rule: a zero total-RBC
ratio with a missing Tier 1 ratio is a placeholder → set to missing. Genuine zero-capital
banks (Tier 1 present) are untouched.

In [14]:
cblr_zero = (panel["RBCRWAJ"] == 0) & panel["RBC1RWAJ"].isna()
panel.loc[cblr_zero, "RBCRWAJ"] = np.nan
print(f"zero-fill placeholders cleared: {cblr_zero.sum():,}")

zero-fill placeholders cleared: 47,077


## PCA tier (regime-switched)

Grades each bank-quarter into 5 capital tiers using the rules in force at that date
(`../literature/pca_label_definition.md`):
- **1990–2014**: 3 ratios, original thresholds
- **2015+**: CET1 added as a 4th ratio, stricter Tier 1 cutoffs
- **2020+ CBLR**: small banks (<$10B) with no risk-based ratios but leverage above the
  year's CBLR cutoff are well capitalized by rule, not missing
- Always: worst ratio sets the tier; `EQV` ≤ 2% → critically undercapitalized

In [15]:
TIER_ORDER = ["well", "adequate", "undercap", "sig_undercap", "crit_undercap"]

post2015 = panel["REPDTE"] >= "2015-01-01"
total, tier1, lev, cet1, eqv = (panel[c] for c in ["RBCRWAJ", "RBC1RWAJ", "RBC1AAJ", "RBCT1CER", "EQV"])

# thresholds: (total, tier1, cet1) — cet1 only binds post-2015; leverage same both regimes
def below(t_total, t1_old, t1_new, t_cet1, t_lev):
    t1_cut = np.where(post2015, t1_new, t1_old)
    cet1_test = post2015 & (cet1 < t_cet1)
    return (total < t_total) | (tier1 < t1_cut) | cet1_test | (lev < t_lev)

has_rbc = total.notna() & tier1.notna() & lev.notna()

# CBLR: qualifying banks deemed well capitalized despite missing RBC ratios
cblr_cutoff = pd.Series(np.nan, index=panel.index)
yr = panel["REPDTE"].dt.year
cblr_cutoff[yr == 2020] = 8.0
cblr_cutoff[yr == 2021] = 8.5
cblr_cutoff[(yr >= 2022) & (panel["REPDTE"] < "2026-07-01")] = 9.0
cblr_cutoff[panel["REPDTE"] >= "2026-07-01"] = 8.0
is_cblr = (~has_rbc) & lev.notna() & (lev > cblr_cutoff) & (panel["ASSET"] < 10_000_000)

conditions = [
    eqv.le(2).fillna(False),
    below(6, 3, 4, 3, 3),
    below(8, 4, 6, 4.5, 4),
    below(10, 6, 8, 6.5, 5),
]
choices = ["crit_undercap", "sig_undercap", "undercap", "adequate"]
panel["pca_tier"] = np.select(conditions, choices, default="well")
panel.loc[~has_rbc, "pca_tier"] = np.nan
panel.loc[is_cblr, "pca_tier"] = "well"
panel["pca_tier"] = pd.Categorical(panel["pca_tier"], categories=TIER_ORDER, ordered=True)

print(panel["pca_tier"].value_counts(dropna=False))
print(f"\nCBLR-deemed-well bank-quarters: {is_cblr.sum():,}")
print(f"unlabelable (no ratios, not CBLR): {panel['pca_tier'].isna().mean():.1%}")

pca_tier
well             1204887
adequate           29889
undercap            8426
crit_undercap       6338
sig_undercap        4884
NaN                 4464
Name: count, dtype: int64

CBLR-deemed-well bank-quarters: 42,613
unlabelable (no ratios, not CBLR): 0.4%


## Onset target

For each currently healthy bank-quarter (well/adequate): does the bank fall to
undercapitalized-or-worse within the next 4 quarters? `onset_4q` = 1 if yes.
`quarters_to_onset` (1–4) records how soon. Rows that are already distressed or
unlabelable get no target (excluded from training).

In [16]:
N = 4

panel = panel.sort_values(["CERT", "REPDTE"]).reset_index(drop=True)
panel["is_distressed"] = panel["pca_tier"].isin(["undercap", "sig_undercap", "crit_undercap"])
panel["is_healthy"] = panel["pca_tier"].isin(["well", "adequate"])

quarters_to_onset = pd.Series(np.nan, index=panel.index)
for k in range(N, 0, -1):
    shifted = panel.groupby("CERT")["is_distressed"].shift(-k).fillna(False).astype(bool)
    quarters_to_onset = quarters_to_onset.where(~shifted, k)

panel["quarters_to_onset"] = quarters_to_onset.where(panel["is_healthy"])
panel["onset_4q"] = np.where(panel["is_healthy"], panel["quarters_to_onset"].notna(), np.nan)

rate = panel.loc[panel["is_healthy"], "onset_4q"].mean()
n_pos = int(panel["onset_4q"].sum())
print(f"trainable rows (healthy): {int(panel['is_healthy'].sum()):,}")
print(f"positives (onset within {N}q): {n_pos:,} ({rate:.2%})")

trainable rows (healthy): 1,234,776
positives (onset within 4q): 9,010 (0.73%)


In [17]:
failures

,ID,RESDATE,QBFDEP,BIDCITY,BIDSTATE,FUND,BIDNAME,PSTALP,FIN,FAILDATE,...,SAVR,RESTYPE1,NAME,CHCLASS1,PTRDATE,COST,QBFASSET,CITY,CERT,FAILYR
0,584,12/31/1980,40680.0,DECATUR,AL,1,CENTRAL BANK OF ALABAMA,AL,2286,1980-12-31,...,FDIC,PA,EAST GADSDEN BANK,NM,0,957.0,43980.0,GADSDEN,40.0,1980
1,1053,8/16/1985,76446.0,ANDALUSIA,AL,1,THE COMMERCIAL BANK,AL,1926,1985-08-16,...,FDIC,A/A,THE COMMERCIAL BANK,NM,0,0.0,77816.0,ANDALUSIA,55.0,1985
2,1420,4/22/1987,11899.0,GERALDINE,AL,1,BANK OF GERALDINE,AL,2680,1987-04-22,...,FDIC,PA,THE PEOPLES BANK,NM,0,934.0,11896.0,COLLINSVILLE,76.0,1987
3,4088,9/23/2016,64713.0,HUNTSVILLE,AR,5,TODAY'S BANK,AR,10522,2016-09-23,...,DIF,PA,ALLIED BANK,SM,0,6213.0,66336.0,MULBERRY,91.0,2016
4,1998,12/9/1988,522990.0,DANIA,FL,1,CITIBANK FLORIDA NA,FL,2941,1988-12-09,...,FDIC,PA,CARIBANK,NM,0,39026.0,525792.0,DANIA,143.0,1988
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,95,NaN,479.0,0,0,1,0,ND,0,1936-12-19,...,FDIC,P&A,NORTHERN AND DAKOTA TRUST COMPANY,NM,0,NaN,NaN,FARGO,NaN,1936
4111,96,NaN,360.0,0,0,1,0,ND,0,1936-12-19,...,FDIC,P&A,FIRST INTERNATIONAL BANK,NM,0,NaN,NaN,MINOT,NaN,1936
4112,97,NaN,763.0,0,0,1,0,ND,0,1936-12-19,...,FDIC,P&A,THE FIRST INTERNATIONAL BANK OF WILLISTON,NM,0,NaN,NaN,WILLISTON,NaN,1936
4113,98,NaN,84.0,0,0,1,0,ND,0,1936-12-21,...,FDIC,P&A,BANK OF BERTHOLD,NM,0,NaN,NaN,BERTHOLD,NaN,1936


## Save the processed panel

Single source of truth for EDA and modeling — downstream notebooks load this file.

In [18]:
out = Path("..") / "data" / "processed" / "panel.parquet"
panel.to_parquet(out)
print(f"saved {panel.shape[0]:,} rows x {panel.shape[1]} cols -> {out}")

saved 1,258,888 rows x 63 cols -> ../data/processed/panel.parquet
